# VWAP/Close Rank Alpha Backtest (Alpaca Historical)

Simple cross-sectional alpha:

- Signal per name: `rank(VWAP / Close)`
- Pipeline: market neutralization + gross scaling via `FinStrat.pass_`
- Execution simulator: `FinBT` (daily rebalance)

This notebook uses `AlpacaHistoricalMarketDataProvider` as the market history source.

In [ ]:
import os

import jax.numpy as jnp
import pandas as pd

from src import (
    AlpacaHistoricalMarketDataProvider,
    FinBT,
    FinStrat,
    finTs,
)
from src.algorithm import cross_section

start_date = "2022-01-01"
end_date = "2025-01-01"
tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "TSLA", "JPM", "XOM", "UNH",
]

# Keep only columns needed by this alpha so backtest starts from first bar.
panel_columns = ("Open", "High", "Low", "Close", "Volume", "VWAP")

if not os.environ.get("APCA_API_KEY_ID") or not os.environ.get("APCA_API_SECRET_KEY"):
    raise RuntimeError("Set APCA_API_KEY_ID and APCA_API_SECRET_KEY in your environment before running.")

provider = AlpacaHistoricalMarketDataProvider(paper=True)

fts = finTs(
    start_date,
    end_date,
    tickers,
    market_data=provider,
    attach_yfinance_classifications=False,
)

In [ ]:


def alpha(panel: jnp.ndarray) -> jnp.ndarray:
    # panel columns are exactly panel_columns order above
    close = panel[:, 3]
    vwap = panel[:, 5]
    raw = vwap / close
    return cross_section.rank(raw)

fs = FinStrat(
    fts,
    alpha,
    neutralization="market",
    panel_columns=panel_columns,
)

bt = FinBT(
    fs,
    fts,
    cash=100_000.0,
    commission=0.0005,
    slippage_pct=0.0005,
).run()

results = bt.results(show=False)
pd.Series(results["metrics"]).to_frame("value")

In [ ]:
results["equity_curve"][["Equity", "DrawdownPct"]].tail()